# Kurtosis and checking normality with Q-Q plots
Kurtosis by hand and with libraries, then Q-Q plots built from percentiles, with scipy, and against a uniform distribution.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import statsmodels.api as sm
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.datasets import load_iris

# fixed seed: the same random data on every run
rng = np.random.default_rng(42)

## 1. Kurtosis by hand
Two seasons with mean 40 and standard deviation 20.

In [ ]:
a = np.array([20, 20, 20, 20, 60, 60, 60, 60])
b = np.array([40, 40, 40, 40, 40, 40, 0, 80])

def kurtosis_by_hand(x):
    # z-scores (std divides by n), then the average of z to the fourth power
    z = (x - x.mean()) / x.std()
    return (z ** 4).mean()

for name, x in [("A", a), ("B", b)]:
    print(name, "mean", x.mean(), "std", x.std(), "kurtosis", kurtosis_by_hand(x))

In [ ]:
# scipy gives excess kurtosis (kurtosis - 3) by default
print(stats.kurtosis(a), stats.kurtosis(b))
print(stats.kurtosis(b, fisher=False))
# pandas: excess kurtosis with a small-sample correction
print(pd.Series(a).kurt(), pd.Series(b).kurt())

## 2. Three types of kurtosis
The generalised normal family: shape 1 is leptokurtic, 2 is the normal, 8 is platykurtic.

In [ ]:
x = np.linspace(-4.5, 4.5, 500)
fig = go.Figure()
for beta, name in [(1, "leptokurtic"), (2, "mesokurtic (normal)"), (8, "platykurtic")]:
    g = stats.gennorm(beta)
    s = g.std()
    excess = float(g.stats(moments="k"))
    # rescale so every curve has standard deviation 1
    fig.add_scatter(x=x, y=g.pdf(x * s) * s, name=f"{name}: {excess:+.2f}")
    # share of the area beyond 3 standard deviations, both sides
    print(name, round(2 * g.sf(3 * s) * 100, 3), "% beyond 3 sd")
fig.update_layout(template="simple_white", xaxis_title="value (sd)", yaxis_title="density")
fig.show()

## 3. Is the iris sepal length normal?
First a histogram, then a Q-Q plot by hand from percentiles.

In [ ]:
sepal = load_iris().data[:, 0]
fig = go.Figure(go.Histogram(x=sepal, nbinsx=20, histnorm="probability density"))
fig.update_layout(template="simple_white", xaxis_title="sepal length (cm)")
fig.show()

In [ ]:
# step 1: theoretical data, 1000 values from a standard normal distribution
theo = rng.normal(0, 1, 1000)
# step 2: the 1st to 99th percentiles of both
pct = np.arange(1, 100)
y_quant = np.percentile(sepal, pct)
x_quant = np.percentile(theo, pct)
# steps 3-4: plot them in pairs
fig = go.Figure(go.Scatter(x=x_quant, y=y_quant, mode="markers"))
fig.update_layout(template="simple_white", xaxis_title="theoretical quantiles", yaxis_title="sepal length quantiles")
fig.show()

The shortcut: `scipy.stats.probplot` sorts the data and computes the normal quantiles itself.

In [ ]:
(osm, osr), (slope, icpt, r) = stats.probplot(sepal, dist="norm")
print("slope", round(slope, 3), "intercept", round(icpt, 3))
fig = go.Figure()
fig.add_scatter(x=osm, y=osr, mode="markers", name="data")
fig.add_scatter(x=osm, y=slope * osm + icpt, mode="lines", name="fitted line")
fig.update_layout(template="simple_white", xaxis_title="theoretical quantiles", yaxis_title="ordered values")
fig.show()

statsmodels computes the same quantiles; `fit=True` standardizes the data, so the diagonal y = x becomes the right reference line. (`sm.qqplot` itself draws with matplotlib, so we plot with Plotly.)

In [ ]:
pp = sm.ProbPlot(sepal, fit=True)
fig = go.Figure()
fig.add_scatter(x=pp.theoretical_quantiles, y=pp.sample_quantiles, mode="markers", name="standardized data")
fig.add_scatter(x=[-3, 3], y=[-3, 3], mode="lines", name="y = x (the '45' line)")
fig.update_layout(template="simple_white")
fig.show()

In [ ]:
# a statistical test of normality: small p-value = evidence against normality
print(stats.shapiro(sepal))

## 4. Q-Q plot against a uniform distribution
The same tool works for any theoretical distribution: pass it with `dist`.

In [ ]:
uni = rng.uniform(0, 1, 1000)
fig = make_subplots(rows=1, cols=3, subplot_titles=["histogram", "against normal", "against uniform"])
fig.add_histogram(x=uni, nbinsx=30, row=1, col=1)
for col, dist in [(2, "norm"), (3, stats.uniform)]:
    (osm, osr), (slope, icpt, r) = stats.probplot(uni, dist=dist)
    fig.add_scatter(x=osm, y=osr, mode="markers", marker_size=3, row=1, col=col)
    fig.add_scatter(x=osm, y=slope * osm + icpt, mode="lines", row=1, col=col)
fig.update_layout(template="simple_white", showlegend=False)
fig.show()
print("excess kurtosis of the uniform sample:", round(stats.kurtosis(uni), 2))